# 관리자(hr_admin) 파이프라인 평가 — Colab Pro+ · Qwen3.5-27B

관리자 골든셋(`eval/golden/admin_golden_set.json`, 133문항)으로 관리자 파이프라인을 **단계별로** 평가해요.

| 단계 | 무엇을 | 정답 필드 | 지표 |
| --- | --- | --- | --- |
| ① 의도·도구 선택 | 질문 → intent · tool · params | `intent` `tool` `params` | 의도 정확도·Macro-F1, 도구 정확도, params 일치율 |
| ② 검색 | rule·both 질문 → 조문 | `gold_chunks.primary` · `must_not_documents` | Recall@5·@30, MRR, 함정 문서 혼입 |
| 집계 조회 | tool·params → 표 | `expected` (건수·ids·총액) | 건수 일치, 명단 일치(F1) |
| ④ 답변 생성 | 조문 + 집계 표 → 답 | `key_facts` · `must_not_include` | 핵심 숫자 포함률, 개인정보 노출 0건, 거절 정확도 |

- 실행: 런타임 → 런타임 유형 변경 → **GPU**(A100 이상 권장) → 위에서부터 차례로
- Colab에는 PostgreSQL이 없어서 HR 데이터(CSV)를 **메모리 SQLite**에 올려 집계 도구를 돌려요.
- 코드를 새로 짜면 **2번 '어댑터' 셀의 함수 4개만** 새 코드에 맞게 바꾸면 나머지는 그대로 돌아요.
- 가이드: `RAG-hyerim/docs/COLAB_QWEN.md`

## 0. 준비 (Colab에서 처음 한 번)

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/hr_chatbot")
    ROOT.mkdir(exist_ok=True)
    for repo in ("backend", "DB", "RAG-hyerim"):
        if not (ROOT / repo).exists():
            subprocess.run(["git", "clone", "-q", f"https://github.com/Jaeyong-s-HR-Chatbot/{repo}.git", str(ROOT / repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "backend" / "requirements.txt")], check=True)
else:   # 로컬: hr_chatbot 폴더 안의 어디서 열어도 됨
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "DB" / "data").is_dir() and (p / "backend").is_dir())
BACKEND, DB, HYERIM = ROOT / "backend", ROOT / "DB", ROOT / "RAG-hyerim"
print(ROOT)

## 1. LLM (vLLM · Qwen3.5-27B)

④ 답변 생성에만 필요해요. ①②·집계만 볼 때는 `USE_LLM = False`로 두면 가짜 LLM으로 흐름만 돌아요.
GPU별 모델은 가이드 1번 표 참고 (A100 40GB → `Qwen/Qwen3.5-27B-FP8`).
Qwen3.5는 Mamba(선형 어텐션) 층이 있어서 동시 요청 수만큼 캐시를 따로 잡아요. 평가는 한 번에 한 문항씩이라 `--max-num-seqs 16`으로 줄여요
(기본 256이면 A100 40GB에서 "max_num_seqs (256) exceeds available Mamba cache blocks" 오류로 시작이 안 돼요).

In [ ]:
USE_LLM = IN_COLAB
MODEL = None          # None이면 GPU 메모리를 보고 자동 선택. 직접 정하려면 예: "Qwen/Qwen3.5-9B"

if USE_LLM:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
    if not gpu:
        raise RuntimeError("GPU가 없어요. 런타임 → 런타임 유형 변경 → GPU(A100 권장)로 바꾸고 다시 실행하세요.")
    gpu_name, mem_gb = gpu.split(",")[0], int(gpu.split(",")[1]) / 1024
    if MODEL is None:
        if mem_gb >= 70:   MODEL, MAX_LEN = "Qwen/Qwen3.5-27B", 16384             # H100 · A100 80GB
        elif mem_gb >= 36: MODEL, MAX_LEN = "Qwen/Qwen3.5-27B-FP8", 8192          # A100 40GB
        elif mem_gb >= 22: MODEL, MAX_LEN = "Qwen/Qwen3.5-27B-GPTQ-Int4", 8192    # L4 24GB
        else:
            raise RuntimeError(f"{gpu_name} {mem_gb:.0f}GB로는 27B를 못 올려요. 런타임 유형에서 A100 또는 L4를 고르세요.")
    else:
        MAX_LEN = 8192
    SERVED = "qwen3.5-27b" if "27B" in MODEL else "qwen3.5-9b"
    print(f"GPU {gpu_name} {mem_gb:.0f}GB → {MODEL} (문맥 {MAX_LEN})")
else:
    SERVED = "fake"

if USE_LLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm"], check=True)
    help_text = subprocess.run(["vllm", "serve", "--help"], capture_output=True, text=True).stdout
    extra = ["--default-chat-template-kwargs", '{"enable_thinking": false}'] if "--default-chat-template-kwargs" in help_text else []
    print("생각 모드 끔" if extra else "생각 모드 끄기 옵션 없음 → LLM_MAX_TOKENS 4096으로 대응")
    vllm = subprocess.Popen(["vllm", "serve", MODEL, "--served-model-name", SERVED, "--max-model-len", str(MAX_LEN),
                             "--gpu-memory-utilization", "0.90", "--max-num-seqs", "16", "--port", "8000", *extra],
                            stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

In [ ]:
import time, urllib.request, json
if USE_LLM:
    for _ in range(120):   # 첫 실행은 모델 다운로드로 오래 걸림 (최대 40분)
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5); print("vLLM 준비됨"); break
        except Exception:
            if vllm.poll() is not None:
                log = open("/content/vllm.log", encoding="utf-8", errors="replace").read().splitlines()
                keys = ("out of memory", "OutOfMemory", "Error:", "ValueError", "not supported", "No available memory", "max_model_len", "CUDA")
                hits = [l for l in log if any(k in l for k in keys) and "Traceback" not in l]
                print("── 원인 후보 (로그에서 오류 줄만) ──"); print("\n".join(hits[-25:]) or "(없음)")
                print("\n── 로그 앞부분 ──"); print("\n".join(log[:40]))
                raise RuntimeError("vLLM이 시작하지 못했어요. 위 '원인 후보'를 확인하세요 (전체 로그: /content/vllm.log)")
            time.sleep(20)
    os.environ.update({"LLM_BASE_URL": "http://127.0.0.1:8000/v1", "LLM_MODEL": SERVED, "LLM_API_KEY": "EMPTY",
                       "LLM_TIMEOUT": "300", "LLM_MAX_TOKENS": str(min(4096, MAX_LEN // 2))})
else:
    os.environ["LLM_BASE_URL"] = ""    # 가짜 LLM

## 2. 데이터·백엔드 불러오기

- 규정 조문: `DB/data/regs` + `DB/data/laws` 청크를 합쳐 검색 대상으로 씀 (DB 없이 keyword 검색)
- HR 데이터: `DB/data/hr/*.csv` → 메모리 SQLite (`hr.` 스키마), 집계 도구의 DB 호출을 여기로 돌림
- 기준일 `AS_OF = 2026-11-09` (골든셋 기대값과 같은 날짜). '올해·이번 달'이 이 날짜 기준이 되도록 도구의 오늘 날짜를 고정

In [ ]:
import csv, re, sqlite3, datetime as dt
from collections import Counter, defaultdict

AS_OF = dt.date(2026, 11, 9)
merged = ROOT / "_admin_eval_chunks.jsonl"
with open(merged, "w", encoding="utf-8") as w:
    for p in (DB / "data" / "regs" / "chunks.jsonl", DB / "data" / "laws" / "chunks.jsonl"):
        w.write(p.read_text(encoding="utf-8").rstrip("\n") + "\n")
os.environ.update({"HRDB_URL": "", "RULES_CHUNKS_PATH": str(merged), "SEARCH_MODE": "keyword", "CHAT_LOG_DB": "false"})

if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
from app import config, db, admin_tools                      # noqa: E402
from app.graph import nodes, retriever                       # noqa: E402
from app.graph.admin_classify import classify_admin          # noqa: E402
print("LLM:", "fake" if config.USE_FAKE_LLM else config.LLM_MODEL, "/ 검색:", retriever.mode())

# HR CSV → 메모리 SQLite
_INT, _FLOAT = re.compile(r"-?\d+"), re.compile(r"-?\d+\.\d+")
def _val(v):
    if v == "": return None
    if _INT.fullmatch(v) and not (len(v) > 1 and v.startswith("0")): return int(v)
    if _FLOAT.fullmatch(v): return float(v)
    return v
con = sqlite3.connect(":memory:", check_same_thread=False)
con.execute("ATTACH DATABASE ':memory:' AS hr")
for f in sorted((DB / "data" / "hr").glob("*.csv")):
    rows = list(csv.reader(open(f, encoding="utf-8-sig")))
    if not rows: continue
    con.execute(f"CREATE TABLE hr.{f.stem} ({', '.join(rows[0])})")
    con.executemany(f"INSERT INTO hr.{f.stem} VALUES ({', '.join('?' * len(rows[0]))})", [[_val(v) for v in r] for r in rows[1:]])
con.row_factory = sqlite3.Row
def _fetch_all(sql, params=()):
    params = tuple(p.isoformat() if isinstance(p, (dt.date, dt.datetime)) else p for p in params)
    return [dict(r) for r in con.execute(sql.replace("%s", "?"), params).fetchall()]
db.fetch_all = _fetch_all
db.fetch_one = lambda sql, params=(): (_fetch_all(sql, params) or [None])[0]

class _FixedDate(dt.date):
    @classmethod
    def today(cls): return AS_OF
admin_tools.date = _FixedDate          # 도구 안의 date.today() → AS_OF

gold = json.loads((HYERIM / "eval" / "golden" / "admin_golden_set.json").read_text(encoding="utf-8"))
ITEMS = gold["items"]
CHUNKS = {}
for l in open(merged, encoding="utf-8"):
    r = json.loads(l); CHUNKS[r["chunk_id"]] = r
print(len(ITEMS), "문항 /", Counter(i["intent"] for i in ITEMS))

## 3. 어댑터 — 평가 대상 시스템과 연결하는 함수 4개

지금은 현재 backend 코드에 연결돼 있어요. **코드를 새로 짜면 이 셀만 바꾸면 돼요.**

| 함수 | 입력 | 출력 |
| --- | --- | --- |
| `route(question)` | 질문 | `{"intent", "tool", "params"}` |
| `search(question, k)` | 질문, 개수 | 조문 ID 목록 (순위순) |
| `run_tool(tool, params)` | 도구 이름, 인자 | `{"건수", "표", ...}` (없는 도구면 `None`) |
| `answer(item, chunk_ids, table)` | 문항, 근거 조문 ID, 집계 표 | 답변 문자열 |

In [ ]:
def route(question):
    r = classify_admin(question, today=AS_OF)
    return {"intent": r.get("intent"), "tool": r.get("admin_tool"), "params": r.get("admin_params") or {}}

def search(question, k=30):
    return [h["id"] for h in retriever.search(question, k=k)]

def run_tool(tool, params):
    if tool not in admin_tools.ADMIN_TOOLS:
        return None
    return admin_tools.run_admin_tool("hr_admin", tool, params or {})

def answer(item, chunk_ids, table):
    chunks = [{"id": c, "citation": f"{CHUNKS[c]['doc_name']} {CHUNKS[c].get('article', '')}", "revised": CHUNKS[c].get("effective_date") or "",
               "text": CHUNKS[c]["text"], "score": 1.0} for c in chunk_ids if c in CHUNKS]
    intent = route(item["question"])["intent"]      # 거절·DB에 없음은 파이프라인처럼 안내 문구로 끝냄
    s = {"employee_id": "E1307", "role": "hr_admin", "masked_question": item["question"], "intent": intent,
         "chunks": chunks, "lookup": {}, "calc": {}}
    if intent in ("refuse", "unavailable"):
        return nodes.refuse(s)["answer"]
    if table is not None:
        s["admin_result"] = table
    return nodes.generate(s)["answer"]

## 4. ① 의도·도구 선택

In [ ]:
def norm(v):
    return str(v.isoformat() if hasattr(v, "isoformat") else v)

R1 = []
for it in ITEMS:
    if it["role"] != "hr_admin":
        continue
    p = route(it["question"])
    gp = it["params"] or {}
    pm = [norm(p["params"].get(k)) == norm(v) for k, v in gp.items()] if it["tool"] and p["tool"] == it["tool"] else []
    R1.append({"id": it["id"], "question": it["question"], "gold_intent": it["intent"], "pred_intent": p["intent"],
               "gold_tool": it["tool"], "pred_tool": p["tool"], "pred_params": p["params"],
               "intent_ok": p["intent"] == it["intent"], "tool_ok": (p["tool"] == it["tool"]) if it["tool"] else None,
               "params_ok": (all(pm) if pm else (False if it["tool"] else None))})

def macro_f1(rows, g, p):
    labels = sorted({r[g] for r in rows})
    f1s = []
    for L in labels:
        tp = sum(r[g] == L and r[p] == L for r in rows); fp = sum(r[g] != L and r[p] == L for r in rows); fn = sum(r[g] == L and r[p] != L for r in rows)
        f1s.append(2 * tp / (2 * tp + fp + fn) if tp else 0.0)
    return sum(f1s) / len(f1s)

rate = lambda xs: round(sum(xs) / len(xs), 3) if xs else None
S1 = {"문항": len(R1), "의도 정확도": rate([r["intent_ok"] for r in R1]), "의도 Macro-F1": round(macro_f1(R1, "gold_intent", "pred_intent"), 3),
      "도구 정확도(data·both)": rate([r["tool_ok"] for r in R1 if r["tool_ok"] is not None]),
      "params 일치율(도구 맞힌 것 중)": rate([r["params_ok"] for r in R1 if r["tool_ok"]])}
print(json.dumps(S1, ensure_ascii=False, indent=1))
print("\n혼동 (정답 → 예측):", Counter((r["gold_intent"], r["pred_intent"]) for r in R1 if not r["intent_ok"]).most_common(10))

## 5. ② 검색 (rule · both)

In [ ]:
R2 = []
for it in ITEMS:
    g = it["gold_chunks"]["primary"]
    if it["intent"] not in ("rule", "both") or not g:
        continue
    got = search(it["question"], 30)
    first = next((i + 1 for i, c in enumerate(got) if c in g), None)
    bad_docs = set(it.get("must_not_documents", []))
    R2.append({"id": it["id"], "recall@5": len(set(g) & set(got[:5])) / len(g), "recall@30": len(set(g) & set(got)) / len(g),
               "mrr": 1 / first if first else 0.0, "trap@5": bool(bad_docs & {c.split("-")[0] for c in got[:5]}) if bad_docs else None, "got": got[:5]})
S2 = {"문항": len(R2), **{k: round(sum(r[k] for r in R2) / len(R2), 3) for k in ("recall@5", "recall@30", "mrr")},
      "함정 문서 혼입": [r["id"] for r in R2 if r["trap@5"]]}
print(json.dumps(S2, ensure_ascii=False, indent=1))

## 6. 집계 조회

- **도구 구현 정확도**: 정답 도구·정답 인자로 돌렸을 때 기대값과 같은가 (도구가 없으면 '미구현')
- **끝까지 정확도**: ①에서 고른 도구·인자로 돌렸을 때 기대값과 같은가

In [ ]:
def compare(res, exp):
    if res is None:
        return {"status": "미구현"}
    out = {"status": "ok", "건수_일치": res.get("건수") == exp.get("건수")}
    if "총액" in exp:
        out["총액_일치"] = res.get("총액") is not None and abs(res["총액"] - exp["총액"]) <= max(1, 0.01 * exp["총액"])
    if "ids" in exp:
        got = {r.get("사번") for r in res.get("표", []) if isinstance(r, dict)}
        want = set(exp["ids"])
        tp = len(got & want)
        out["명단_F1"] = round(2 * tp / (len(got) + len(want)), 3) if (got or want) else 1.0
    return out

R3 = []
pred = {r["id"]: r for r in R1}
for it in ITEMS:
    if not it["tool"] or not it["expected"]:
        continue
    try:
        impl = compare(run_tool(it["tool"], it["params"]), it["expected"])
    except Exception as e:
        impl = {"status": f"오류 {type(e).__name__}"}
    p = pred.get(it["id"])
    try:
        e2e = compare(run_tool(p["pred_tool"], p["pred_params"]), it["expected"]) if p and p["pred_tool"] else {"status": "도구 선택 안 함"}
    except Exception as e:
        e2e = {"status": f"오류 {type(e).__name__}"}
    R3.append({"id": it["id"], "tool": it["tool"], "impl": impl, "e2e": e2e})

impl_ok = [r for r in R3 if r["impl"]["status"] == "ok"]
S3 = {"기대값 있는 문항": len(R3), "미구현 도구": sorted({r["tool"] for r in R3 if r["impl"]["status"] == "미구현"}),
      "구현 도구 건수 일치율": rate([r["impl"]["건수_일치"] for r in impl_ok]),
      "구현 도구 명단 F1 평균": rate([r["impl"]["명단_F1"] for r in impl_ok if "명단_F1" in r["impl"]]),
      "끝까지 건수 일치율": rate([r["e2e"].get("건수_일치", False) for r in R3])}
print(json.dumps(S3, ensure_ascii=False, indent=1))

## 7. ④ 답변 생성

노션 '평가' 방식과 같아요: 근거 = **정답 조문 + 검색 상위의 비슷한 오답 조문으로 5개**, 집계 표 = **기대값**(정답 표).
모델이 근거와 표를 제대로 읽고 답하는지만 봐요. 프롬프트 조정은 `SPLIT="dev"`, 최종 비교는 `"test"` 한 번만.

In [ ]:
SPLIT, LIMIT = "dev", 20          # 전체: LIMIT = None
REFUSAL = ("확인할 수 없", "조회할 수 없", "제공하지 않", "알려드릴 수 없", "DB에 없", "없어요", "인사시스템", "권한")
PII = [r"\d{6}-?[1-4]\d{6}", r"01[016789]-?\d{3,4}-?\d{4}", r"\d{2,6}-\d{2,6}-\d{4,8}"]
NUM = re.compile(r"(\d[\d,]*(?:\.\d+)?)\s*(억|만|천)?")

def nums(text):
    out = []
    for m in NUM.finditer(text or ""):
        v = float(m.group(1).replace(",", "")) * {"억": 1e8, "만": 1e4, "천": 1e3, None: 1}[m.group(2)]
        out.append(v)
    return out

targets = [it for it in ITEMS if it["role"] == "hr_admin" and it["split"] == SPLIT and (it["key_facts"] or it["intent"] in ("unavailable", "refuse"))][:LIMIT]
R4 = []
for n, it in enumerate(targets, 1):
    gold_c = it["gold_chunks"]["primary"][:5]
    ctx = gold_c + [c for c in search(it["question"], 15) if c not in gold_c][: 5 - len(gold_c)] if it["intent"] in ("rule", "both") else []
    t = time.time()
    try:
        ans = answer(it, ctx, it["expected"])
    except Exception as e:
        ans = f"[오류] {type(e).__name__}: {e}"
    ms = int((time.time() - t) * 1000)
    a_nums = nums(ans)
    fact_hits = []
    for f in it["key_facts"]:
        fn = [x for x in nums(f) if x >= 2]
        fact_hits.append(all(any(abs(x - y) <= max(0.01 * x, 0.5) for y in a_nums) for x in fn) if fn else None)
    num_facts = [h for h in fact_hits if h is not None]
    R4.append({"id": it["id"], "intent": it["intent"], "question": it["question"], "answer": ans, "ms": ms,
               "숫자_key_facts_포함률": rate(num_facts), "pii_노출": any(re.search(p, ans) for p in PII),
               "거절함": any(h in ans for h in REFUSAL)})
    print(f"[{n}/{len(targets)}] {it['id']} {ms}ms")

refuse_items = [r for r in R4 if r["intent"] in ("unavailable", "refuse")]
lat = sorted(r["ms"] for r in R4)
S4 = {"문항": len(R4), "숫자 key_facts 포함률": rate([r["숫자_key_facts_포함률"] for r in R4 if r["숫자_key_facts_포함률"] is not None]),
      "개인정보 노출 건수": sum(r["pii_노출"] for r in R4), "거절 정확도(unavailable·refuse)": rate([r["거절함"] for r in refuse_items]),
      "응답시간 p50(ms)": lat[len(lat) // 2] if lat else None}
print(json.dumps(S4, ensure_ascii=False, indent=1))

숫자가 없는 key_facts(절차·조건 문장)는 규칙으로 채점이 안 돼요. 아래 표에서 답변을 직접 보거나, 최종 비교 때 Opus 의미 판정(`RAG-hyerim/scripts/eval_golden.py`의 judge 방식)을 붙이세요.

In [ ]:
for r in R4[:5]:
    print(f"--- {r['id']} [{r['intent']}] {r['question']}\n{r['answer'][:500]}\n")

## 8. 결과 저장

In [ ]:
out_name = f"admin_eval_{dt.datetime.now():%Y%m%d_%H%M}_{'fake' if config.USE_FAKE_LLM else SERVED}"
out_dir = (Path("/content/drive/MyDrive/hr_chatbot_eval") if IN_COLAB else HYERIM / "eval" / "golden" / "runs") / out_name
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "summary.json").write_text(json.dumps({"①": S1, "②": S2, "집계": S3, "④": S4}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
for name, rows in (("route", R1), ("search", R2), ("tools", R3), ("answers", R4)):
    (out_dir / f"{name}.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False, default=str) + "\n" for r in rows), encoding="utf-8")
print("저장:", out_dir)